In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions

# -----------------------------
# 1️⃣ Embedding function
# -----------------------------
class MyEmbeddingFunction:
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

    # Called when adding documents
    def __call__(self, input):
        if isinstance(input, list):
            return self.model.encode(input).tolist()
        return self.model.encode([input]).tolist()[0]

    # Called when querying
    def embed_query(self, input):
        if isinstance(input, list):
            return self.model.encode(input).tolist()
        return self.model.encode([input]).tolist()[0]


c:\Multi-Document RAG System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -----------------------------
# 2️⃣ Load PDFs
# -----------------------------
def load_documents(path):
    documents = []

    if os.path.isfile(path) and path.endswith(".pdf"):
        loader = PyPDFLoader(path)
        documents.extend(loader.load())
    elif os.path.isdir(path):
        for filename in os.listdir(path):
            if filename.endswith(".pdf"):
                file_path = os.path.join(path, filename)
                print(f"📄 Loading: {filename}")
                loader = PyPDFLoader(file_path)
                documents.extend(loader.load())
    else:
        print(f"❌ No PDFs found at {path}")

    return documents


In [3]:
# 3️⃣ Split text into chunks
# -----------------------------
def split_text(documents, chunk_size=300, chunk_overlap=50):
    chunks = []
    for doc in documents:
        text = doc.page_content
        start = 0
        while start < len(text):
            end = min(start + chunk_size, len(text))
            chunk = text[start:end]
            chunks.append(chunk)
            start += chunk_size - chunk_overlap
    return chunks


In [4]:
# -----------------------------
# 4️⃣ Add chunks to Chroma collection
# -----------------------------
def create_vector_store(chunks, collection, source_name="document.pdf"):
    print("\n--- First 5 chunks being added to vector DB ---")
    for i, chunk in enumerate(chunks[:5]):
        print(f"Chunk {i} | length: {len(chunk)} | metadata: {{'source': '{source_name}'}}\n{chunk[:200]}...\n")

    ids = [f"doc_{i}" for i in range(len(chunks))]
    metadatas = [{"source": source_name} for _ in chunks]

    collection.add(
        documents=chunks,
        metadatas=metadatas,
        ids=ids
    )


In [5]:
# 5️⃣ Query RAG system
# -----------------------------
def query_rag_system(query, vector_store, n_results=1):
    results = vector_store.query(query_texts=[query], n_results=n_results)
    if results and results['documents'][0]:
        return results['documents'][0][0]
    return "No relevant information found."


In [ ]:
# 6️⃣ Main program
# -----------------------------
def main():
    path = r"C:\Multi-Document RAG System\1706326973548.pdf"  # single PDF or folder

    # Initialize Chroma client
    client = chromadb.Client()

    # Collection name
    collection_name = "rag_docs"

    # Load or create collection
    if collection_name in [c.name for c in client.list_collections()]:
        print("📦 Loading existing vector DB...")
        vector_store = client.get_collection(name=collection_name)
    else:
        print("📦 No vector DB found. Creating one...")
        embedding_function = MyEmbeddingFunction()
        vector_store = client.create_collection(
            name=collection_name,
            embedding_function=embedding_function
        )

        # Load documents
        docs = load_documents(path)
        if not docs:
            print("❌ No documents found. Exiting.")
            return

        # Split text into chunks
        chunks = split_text(docs)

        # Add chunks to collection
        create_vector_store(chunks, vector_store, source_name=os.path.basename(path))
        print("✅ Vector database created")

    # Interactive Q&A loop
    while True:
        query = input("\n❓ Ask a question (or type 'exit'): ")
        if query.lower() == "exit":
            print("👋 Exiting RAG system. Goodbye!")
            break

        print("🤔 Thinking...")
        answer = query_rag_system(query, vector_store)
        print("\n🧠 Answer:\n", answer)


if __name__ == "__main__":
    main()

📦 No vector DB found. Creating one...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5177.84it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- First 5 chunks being added to vector DB ---
Chunk 0 | length: 37 | metadata: {'source': '1706326973548.pdf'}
DATA SCIENCE
INTERVIEW
GUIDE
ACE-PREP...

Chunk 1 | length: 226 | metadata: {'source': '1706326973548.pdf'}
ABOUT THE AUTHOR
ACE-PREP
ACE PREP are researchers
Based in London, England.
The ACE-PREP is a collective; we work with the
most senior academic researchers, writers and
knowledge makers.
We are in th...

Chunk 2 | length: 300 | metadata: {'source': '1706326973548.pdf'}
© Copyright 2022 by (United Arts Publishing, England.) - All rights reserved.
This document is geared towards providing exact and reliable information in regards to
the topic and issue covered. The pu...

Chunk 3 | length: 300 | metadata: {'source': '1706326973548.pdf'}
 is
not required to render accounting, officially permitted, or otherwise, qualified services.
If advice is necessary, legal or professional, a practised individual in the profession
should be ordered...

Chunk 4 | length: 300 | metadat